#### Load Data and Define Features

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
import joblib

df = pd.read_csv('../data/prepared_energy_dataset.csv')
train_df = df[df['split'] == 'train'].copy()
val_df = df[df['split'] == 'val'].copy()
test_df = df[df['split'] == 'test'].copy()

targets = ['Total_Power_Watt', 'Predicted_Energy_Requirement_kWh', 'Energy_Management_Status']

features_df = train_df.drop(columns=targets + ['Timestamp', 'split'])
categorical_cols = features_df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = [c for c in features_df.columns if c not in categorical_cols + ['Hour', 'Day_of_Week', 'Month', 'Is_Weekend', 'Peak_Time']]

print(f"Categorical features: {len(categorical_cols)}")
print(f"Continuous Numerical features: {len(numerical_cols)}")

Categorical features: 6
Continuous Numerical features: 24


/var/folders/t9/5qyb5pw554s_w_jt9c13myrh0000gn/T/ipykernel_48200/824794199.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = features_df.select_dtypes(include=['object']).columns.tolist()


#### Outlier & Skewness Analysis

In [ ]:
Q1 = train_df[numerical_cols].quantile(0.25)
Q3 = train_df[numerical_cols].quantile(0.75)
IQR = Q3 - Q1

outliers_pct = ((train_df[numerical_cols] < (Q1 - 1.5 * IQR)) | (train_df[numerical_cols] > (Q3 + 1.5 * IQR))).mean() * 100
skewness = train_df[numerical_cols].skew()

outlier_analysis = pd.concat([outliers_pct.rename('% IQR outliers (train)'), skewness.rename('skewness')], axis=1)
display(outlier_analysis.sort_values(by='% IQR outliers (train)', ascending=False).head(10).round(2))
target_b = 'Predicted_Energy_Requirement_kWh'
target_b_hi = train_df[target_b] > (train_df[target_b].quantile(0.75) + 1.5 * (train_df[target_b].quantile(0.75) - train_df[target_b].quantile(0.25)))

print(f"\n{target_b}: {(target_b_hi).mean()*100:.1f}% above the upper fence.")
print(f"Those rows have mean traffic {train_df.loc[target_b_hi, 'Traffic_Volume_Mbps'].mean():.0f} Mbps vs {train_df['Traffic_Volume_Mbps'].mean():.0f} overall.")

,% IQR outliers (train),skewness
Packet_Loss_Rate_Pct,12.24,6.61
Signaling_Message_Count,5.89,2.41
Handover_Request_Count,5.28,2.18
Session_Request_Count,5.15,2.05
Uplink_Traffic_Mbps,4.44,1.95
Active_User_Count,4.04,1.71
Packet_Arrival_Rate_Kpps,3.90,1.66
Control_Plane_Load_Pct,3.81,1.45
Downlink_Traffic_Mbps,3.64,1.45
Throughput_Mbps,3.16,1.38



Predicted_Energy_Requirement_kWh: 6.6% above the upper fence.
Those rows have mean traffic 1238 Mbps vs 3615 overall.


#### Construct the Preprocessing Pipeline

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log1p', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols)
    ], 
    remainder='passthrough'
)

print("Pipeline configured successfully.")

Pipeline configured successfully.


#### We fit the pipeline exclusively on the training set and transform all sets

In [ ]:
X_train = train_df.drop(columns=targets + ['Timestamp', 'split'])
X_val = val_df.drop(columns=targets + ['Timestamp', 'split'])
X_test = test_df.drop(columns=targets + ['Timestamp', 'split'])

y_train_power = train_df['Total_Power_Watt']
y_val_power = val_df['Total_Power_Watt']
y_test_power = test_df['Total_Power_Watt']

X_train_processed = pd.DataFrame(preprocessor.fit_transform(X_train), index=X_train.index, columns=preprocessor.get_feature_names_out())
X_val_processed = pd.DataFrame(preprocessor.transform(X_val), index=X_val.index, columns=preprocessor.get_feature_names_out())
X_test_processed = pd.DataFrame(preprocessor.transform(X_test), index=X_test.index, columns=preprocessor.get_feature_names_out())

X_train_processed.to_csv('../data/X_train_processed.csv', index=False)
X_val_processed.to_csv('../data/X_val_processed.csv', index=False)
X_test_processed.to_csv('../data/X_test_processed.csv', index=False)

y_train_power.to_csv('../data/y_train_power.csv', index=False)
y_val_power.to_csv('../data/y_val_power.csv', index=False)
y_test_power.to_csv('../data/y_test_power.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor_pipeline.pkl')

print(f"Processing complete. Final feature count: {X_train_processed.shape[1]}")
print("Artefacts saved to /data and /models.")

Processing complete. Final feature count: 10549
Artefacts saved to /data and /models.
